# TumbleScope · first synthetic training run

Select **Runtime → Change runtime type → GPU** on the free tier, then run the cells in order. Upload only the verified CP2 bundle. On reconnection reopen this notebook, select a GPU, upload the same bundle and mount the same Drive. The trainer validates and resumes the latest completed epoch. All checkpoints remain individually accessible in your Drive.

In [ ]:
import hashlib
import json
import os
import platform
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import torch
from google.colab import files, drive

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. Select a free GPU runtime and rerun. No training has started.")
print("GPU", torch.cuda.get_device_name(), "PyTorch", torch.__version__, "CUDA", torch.version.cuda,
      "Python", platform.python_version())

Choose the locally generated **cp2_colab.zip**. This notebook checks its published digest before loading any Python code.

In [ ]:
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Select one CP2 bundle.")
raw = next(iter(uploaded.values()))
expected = "cedd0fb8638fbeaf21a46e24f216b2d176cf6d79710e2b67d6c0a5224efa5a0a"
if hashlib.sha256(raw).hexdigest() != expected:
    raise ValueError("Wrong or modified bundle.")
work = Path(tempfile.mkdtemp(prefix="tumblescope-", dir="/content"))
bundle = work / "cp2_colab.zip"
bundle.write_bytes(raw)
del uploaded, raw
code = {"ml/model.py", "ml/train.py", "ml/bundle.py"}
payload = code | {"config.json", "train.npz", "val.npz"}
with zipfile.ZipFile(bundle) as archive:
    members = archive.namelist()
    if len(members) != len(payload)+1 or set(members) != payload | {"manifest.json"}:
        raise ValueError("Unexpected or missing bundle member")
    hashes = json.loads(archive.read("manifest.json"))
    if set(hashes) != payload:
        raise ValueError("Invalid manifest")
    for name in sorted(payload):
        data = archive.read(name)
        if hashlib.sha256(data).hexdigest() != hashes[name]:
            raise ValueError(f"Checksum mismatch: {name}")
        if name in code:
            target = work / name
            target.parent.mkdir(exist_ok=True)
            target.write_bytes(data)
(work / "ml/__init__.py").touch()
print("Verified synthetic development bundle", expected)

**Persistent checkpoints.** Run the next cell and explicitly authorize mounting your own Google Drive. Only training outputs are saved there. The uploaded bundle stays in the temporary Colab session. Use one active training session for this run.

In [ ]:
drive.mount("/content/drive")
out = Path("/content/drive/MyDrive/TumbleScope") / ("cp2-" + expected[:16])
print("Persistent output directory", out)

**Launch or resume.** This cell validates the arrays and split counts, then runs the repository trainer. The first real CUDA parameter update writes `gpu_launch.json`. Each completed epoch writes a checkpoint, checksum, training history and latest-epoch pointer to the mounted Drive.

In [ ]:
subprocess.run([sys.executable, "-m", "ml.train", "colab", str(bundle), str(out)],
               cwd=work, check=True)

**Download.** Choose any completed epoch or the final checkpoint. Each download ZIP includes its checksum. Extract the final download locally into `models/` (the .pt and .sha256 files belong together). Keep the Colab log with your run evidence. Run the local acceptance command printed below to create the repository's measured training log. You can also retrieve any completed epoch directly from the displayed Drive folder after a disconnection.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
latest = json.loads((out / "latest.json").read_text())
print("Latest completed epoch", latest)
options = sorted(p.name for p in out.glob("*.pt") if p.with_suffix(".sha256").exists())
selection = widgets.Dropdown(options=options, value="cnn_v1.pt" if "cnn_v1.pt" in options else latest["file"])
button = widgets.Button(description="Download checkpoint")
def download(_):
    checkpoint = out / selection.value
    checksum = checkpoint.with_suffix(".sha256")
    if hashlib.sha256(checkpoint.read_bytes()).hexdigest() != checksum.read_text().strip():
        raise ValueError("Stored checkpoint checksum mismatch")
    target = work / (checkpoint.stem + "-download.zip")
    with zipfile.ZipFile(target, "w") as result:
        for path in (checkpoint, checksum, out/"cnn_training_log.json", out/"gpu_launch.json"):
            result.write(path, path.name)
    files.download(str(target))
button.on_click(download)
display(selection, button)
print(r".\.venv\Scripts\python.exe -m ml.train accept data/synthetic/cp2_colab.zip models/cnn_v1.pt")